# 月データでムーンベースの場所を決めよう（データ分析と探究活動）

月の公開データを使って、**月面基地をどこに建てるか**を自分で決めます。
コードを書く必要はありません。セルを上から順に実行し、`# ★ここを変える` と書いてある
数字だけを書き換えて、結果をワークシートに記録していきます。

## あなたのミッションを1つ選ぶ（ワークシートに○）

| ミッション | 基地に必要なこと |
|---|---|
| ☀️ **太陽光発電基地** | 一年中よく日が当たること（発電したい） |
| 🧊 **氷採掘基地** | 氷がありそうなこと（＝ずっと日が当たらない「永久影」） |
| 🏠 **有人基地（人が住む）** | 電力も使えて、かつ1日の温度変化がおだやかなこと |

同じデータでも、ミッションが変われば「最適な場所」は変わります。

## 進め方
1. 上から順にセルを実行する（Colab なら「ランタイム」→「すべてのセルを実行」）
2. 各ステップで、まずワークシートに **予想** を書く
3. `# ★ここを変える` の数字を書き換えて実行し、結果をワークシートに記録
4. **気づいたこと** を書く

In [ ]:
# 準備：ヘルパー（moonkit）を読み込む
from moonkit import *
for k in DATASETS:
    print(k, ':', len(load(k)), '地点')

---
## ステップ1：月の温度は1日でどれくらい変わる？

月には空気（大気）がありません。空気がないと、温度はどうなるでしょう？

**予想をワークシートに書いてから**、下のセルを実行します。

In [ ]:
my_band = (-10, 10)      # ★ここを変える：調べたい緯度の範囲（例：赤道なら (-10, 10)、緯度45度なら (40, 50)）

band = region(load('温度'), lat=my_band)
diurnal_curve(band)                       # 1日の温度変化カーブ
band = daily_swing(band)                  # 各地点の「1日の平均・較差・ばらつき」を計算
summary(band, 't_mean_K', 't_swing_K', 't_std_K')
#   t_mean_K  … 1日の平均温度
#   t_swing_K … 1日の温度差（いちばん暑い時 − いちばん寒い時）
#   t_std_K   … 温度のばらつき（分散の平方根）。カーブが大きく上下するほど大きい

**ワークシートに記録**：1日の温度差（`t_swing_K` の平均）は何 K？　地球の砂漠の昼夜差はせいぜい 20〜30 ℃ です。

**気づいたこと**：なぜこんなに差が大きいの？　カーブの形（朝の上がり方と、夕方〜夜の下がり方）は左右対称？

---
## ステップ2：月の「海」と「陸」で何が違う？

月を見ると、黒っぽく平らな「海（マリア）」と、白っぽくでこぼこの「陸（高地）」があります。
この2つは、クレーターのでき方が違います。

**予想**：海と陸、どちらがクレーターが多いと思う？

In [ ]:
# まず全体のクレーターの密度を地図で見る
grid_count(load('クレーター'), lat_step=10, lon_step=10)

In [ ]:
# 「海」と「陸」の代表的な場所を四角で指定して、クレーターの数を比べる
umi_riku = {
    '海': [{'lat': (20, 50), 'lon': (-40, 5)},    # 雨の海
           {'lat': (15, 40), 'lon': (5, 35)},     # 晴れの海
           {'lat': (-5, 20), 'lon': (18, 45)},    # 静かの海
           {'lat': (-10, 40), 'lon': (-80, -30)}, # 嵐の大洋
           {'lat': (7, 27), 'lon': (48, 70)}],    # 危難の海
    '陸': [{'lat': (-55, -25), 'lon': (-20, 40)}, # 南の高地
           {'lat': (-30, 30), 'lon': (120, 175)}, # 裏側
           {'lat': (30, 60), 'lon': (100, 160)}], # 北東の高地
}
c = classify_by_box(load('クレーター'), umi_riku)
print(c['区分'].value_counts())              # 海・陸それぞれのクレーターの数
summary_by(c, group='区分', value='diam_km')  # 海・陸それぞれの直径の平均など

**ワークシートに記録**：海と陸のクレーターの数、直径の平均。

**気づいたこと**：海のほうがクレーターが少ないのはなぜ？（ヒント：海は昔どろどろに溶けた溶岩でおおわれた）

---
## ステップ3：月の南極を細かく見る

ステップ1で、極に近いほど温度変化がおだやかだと分かりました。
基地の候補として、**南極**をくわしく見ます。南極には「一年中日が当たらない場所（永久影）」と
「ほぼずっと日が当たる場所」が、となり合って存在します。

**予想**：南極の「平均日照率」のヒストグラムは、どんな形になると思う？

In [ ]:
極 = south_pole(load('極域日照'))          # 南緯80度より南
my_threshold = 5                          # ★ここを変える：「これより日照率が低ければ永久影だろう」というしきい値 [%]

hist(極, 'average_illumination_percent', vline=my_threshold)
暗い場所 = 極[極['average_illumination_percent'] <= my_threshold]
print('しきい値', my_threshold, '% 以下の地点：', len(暗い場所), '個')
scatter(暗い場所, 'lon', 'lat')           # その場所を地図に表示

**ワークシートに記録**：自分で決めたしきい値と、それに当てはまった地点の数。

**気づいたこと**：しきい値を大きく（ゆるく）すると、地点の数はどう変わる？

---
## ステップ4：あなたのミッションの基地はどこ？

複数の条件を「0〜1の点数」に直して重みをつけて合計し、点数の高い場所を探します。
**自分のミッションに合わせて `want` を書き換えます。**

| ミッション | おすすめの `want` |
|---|---|
| ☀️ 太陽光発電基地 | `{'average_illumination_percent': ('高い', 1)}` |
| 🧊 氷採掘基地 | `{'permanent_shadow_fraction': ('高い', 1)}` |
| 🏠 有人基地 | `{'average_illumination_percent': ('高い', 1), 't_swing_K': ('低い', 1)}` |

重み（かっこ内の数字）を変えると、どの条件を重く見るかを調整できます。

In [ ]:
# 南極の各地点に「1日の温度差（t_swing_K）」の情報もくっつける
極 = join_grid(south_pole(load('極域日照')), daily_swing(load('温度')), ['t_swing_K'])

want = {                                              # ★ここを変える：上の表から自分のミッションのものを写す
    'average_illumination_percent': ('高い', 1),
    't_swing_K':                    ('低い', 1),
}

best = site_score(極, want, top=10)
print('あなたのミッションでの上位10地点：')
print(best[['lat', 'lon', 'average_illumination_percent',
            'permanent_shadow_fraction', 't_swing_K', 'スコア']].round(2).to_string(index=False))

# 全地点をスコアで色分けした地図（明るいほど条件に合う）
scatter(site_score(極, want, top=None), 'lon', 'lat', color='スコア')

**ワークシートに記録**：上位に出た場所の緯度・経度、そこの日照率・永久影率・温度差。
そして「**ここに基地を建てる**」と決めた1地点と、その**理由（3つ以上の文で）**。

---
## ステップ5：他の班と比べる

- 他の班（ちがうミッション）が選んだ場所は、あなたの場所とどれくらい離れている？
- なぜ違う場所になった？　同じデータを使っているのに。
- 「1つの正解」はある？　それとも「目的によって最適地は変わる」？

ここはコードなし。ワークシートに書いて、クラスで共有します。

---
## （発展・任意）ステップ6：機械が決めた基準と、自分が決めた基準を比べる

ステップ3で、あなたは「日照率がこれ以下なら永久影」というしきい値を**自分で**決めました。
同じことを、機械学習に**学習させる**とどうなるでしょう。決定木・ニューラルネットなど
5種類のモデルを取り替えて、境界線の形と「あたりやすさ」「読みやすさ」を比べます。

このステップは別ノートブック `course_moonbase_ml.ipynb` で行います。時間が
余った班・興味のある人向けで、必須ではありません。